<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/07_01_ggs416_26_03_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ GGS416 Week 7: Intro to object detection 🛰️

This week introduces object detection using a pretrained You Only Look Once (YOLO) model and high resolution aerial imagery.

The goal is to keep the workflow understandable, even if you are still new to Python. At the end of the code intro, you will have a set of re-usable functions that will enable you to go off and explore the use of a computer vision model on satellite imagery.

We will build the workflow step by step as follows:

1. understand what object detection is
2. import packages in small groups
3. download and clip a NAIP image
4. prepare image tiles
5. run a pretrained YOLO model
6. interpret the detections
7. apply the same workflow to a new place as the class exercise


## Learning objectives

By the end of class, students should be able to:

1. Explain the difference between segmentation, classification, and object detection.
2. Identify which Python packages are used for imagery access, mapping, and detection.
3. Download and clip a NAIP image from the Planetary Computer.
4. Explain why we tile large aerial images before detection.
5. Run a pretrained YOLO model and inspect the quality of the bounding boxes it returns.
6. Reuse the workflow in a new location as an exercise.


## Lesson outline

The suggested lesson outline and timing is as follows:
- Conceptual introduction to object detection and YOLO - 10 minutes
- Downloading and clipping NAIP imagery - 10 minutes
- Tiling imagery and preparing YOLO input - 10 minutes
- Running the pretrained model and reading detections - 10 minutes
- Guided class exercise for a new location - 30 minutes


## Install packages

Run this only if the packages are not already installed in your environment.

In [ ]:
#!pip -q install pystac-client planetary-computer requests rasterio matplotlib pandas pillow ultralytics

## What is object detection?

In remote sensing, there are three common prediction tasks:

- Segmentation can provide a label for every pixel.
- Classification can provide a label for each image.
- Object detection provides a class label, a confidence score, and a bounding box for each object.

The You Only Look Once (YOLO) model is a fast object detection model that predicts object locations and labels in one pass through the image.

For this introductory class, YOLO is useful because it can often detect classes such as:
- car
- truck
- bus
- boat
- airplane

But it is important to note what a generic pretrained model usually does not do well in overhead imagery:
- building footprints
- roads
- runways
- specialized remote sensing classes unless the model was trained on them

As today is more about getting started, we will not be training our own model.

Instead, make sure you download the `yolov8n.pt` model weights from the class GitHub repo, and import it into this workbook (this saves quite a few steps!).


## Import the packages for basic Python work

We start with very common Python packages. These are not specific to geospatial data or machine learning. You should already be familiar with the following:

- `os` helps us work with files and folders.
- `numpy` helps us work with image arrays.
- `pandas` helps us organize detections in tables.


In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)

## Import the packages for display and plotting

Next we import packages used to show imagery and draw bounding boxes.

- `matplotlib.pyplot` displays images and charts.
- `matplotlib.patches` lets us draw rectangles around detected objects.
- `display` helps us show tables nicely in a notebook.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

## Import the packages for satellite imagery

These packages are the geospatial part of the workflow.

- `pystac_client` searches the Planetary Computer catalog.
- `planetary_computer` signs the image links so we can access them.
- `rasterio` reads raster imagery such as NAIP.
- `from_bounds` and `transform_bounds` help us clip the exact geographic area we want.


In [ ]:
from pystac_client import Client
import planetary_computer as pc
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

## Import the package for object detection

Now we add the machine learning package.

- `YOLO` from `ultralytics` loads a pretrained detection model and runs inference on images.

See here for more information [https://github.com/ultralytics/ultralytics](https://github.com/ultralytics/ultralytics)

In [ ]:
from ultralytics import YOLO

## First set of helper functions: display an RGB image clearly

Satellite imagery often contains large data values. Before plotting, we usually stretch the image so it looks better on screen.

The first function below does a simple percentile stretch. The second function displays an image with a title.

In [ ]:
def stretch_rgb(rgb_bands):
    """Convert a 3-band image from bands-first to bands-last and stretch it for display."""
    rgb_hwc = np.moveaxis(rgb_bands, 0, -1).astype("float32")
    p2 = np.nanpercentile(rgb_hwc, 2)
    p98 = np.nanpercentile(rgb_hwc, 98)
    return np.clip((rgb_hwc - p2) / (p98 - p2 + 1e-6), 0, 1)


def show_image(img, title, figsize=(7, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

## Second set of helper functions: search for a NAIP image

We now write a small function that searches the Planetary Computer for NAIP imagery.

The function does three things:
1. searches for NAIP scenes that intersect our bounding box
2. keeps the most recent date
3. chooses the scene with the greatest overlap with our area of interest


In [ ]:
def bbox_overlap_area(a, b):
    """Calculate the overlap area of two lon/lat bounding boxes."""
    minx = max(a[0], b[0])
    miny = max(a[1], b[1])
    maxx = min(a[2], b[2])
    maxy = min(a[3], b[3])
    if maxx <= minx or maxy <= miny:
        return 0.0
    return (maxx - minx) * (maxy - miny)


def get_latest_naip_item(bbox, datetime="2021-01-01/2024-12-31"):
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    items = list(
        catalog.search(
            collections=["naip"],
            bbox=bbox,
            datetime=datetime,
            limit=12,
            method="POST",
        ).items()
    )

    if not items:
        raise ValueError("No NAIP scenes found for the requested bbox and date range.")

    latest_date = max(item.datetime for item in items if item.datetime).date()
    same_day_items = [item for item in items if item.datetime and item.datetime.date() == latest_date]
    return max(same_day_items, key=lambda item: bbox_overlap_area(bbox, item.bbox))

## Third set of helper functions: clip the image to our study area

The NAIP image is stored in a projected coordinate system, but our bounding box is written in longitude and latitude.

So we have to:
1. convert the lon/lat bounding box into the raster's coordinate system
2. build a raster window from those bounds
3. read only the 3 RGB bands we need


In [ ]:
def read_naip_clip(item, bbox):
    href = item.assets["image"].href

    with rasterio.open(href) as src:
        minx, miny, maxx, maxy = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(minx, miny, maxx, maxy, src.transform)
        rgb = src.read([1, 2, 3], window=window)

    if rgb.shape[1] == 0 or rgb.shape[2] == 0:
        raise ValueError("The requested bbox produced an empty image. Try a different location.")

    return rgb

## Choose a place to analyze

We start with a tight airport area because large parked airplanes are relatively easy for a generic pretrained YOLO model to detect from overhead imagery.

In [ ]:
site_name = "Dulles Airport"
bbox = (-77.4555, 38.944, -77.448, 38.950)

## Find the image and inspect its metadata

This cell searches the catalog and prints the date of the image we selected.

In [ ]:
item = get_latest_naip_item(bbox)
print("Selected site:", site_name)
print("Selected NAIP acquisition date:", item.datetime.date())
print(item.id)

## Read and display the clipped image

Now we read the image into memory, stretch it for display, and show it.

Notice the data shape before and after stretching:
- raw raster: `(bands, rows, cols)`
- display image: `(rows, cols, bands)`

In [ ]:
rgb = read_naip_clip(item, bbox)
rgb_display = stretch_rgb(rgb)

print("Raw raster shape:", rgb.shape)
print("Display image shape:", rgb_display.shape)
show_image(rgb_display, f"NAIP clip: {site_name}")

## Why do we tile the image?

Large aerial images are usually too large to process at full size with a detector.

We create smaller image tiles because:
- the model expects images near a standard size such as 640 by 640 pixels
- small targets like vehicles and aircraft are easier to detect if we do not shrink the whole scene too much
- tiling makes it easier to scale the workflow to larger study areas


## Fourth set of helper functions: create image tiles

The first small function calculates where each tile should begin. The second function actually cuts the image into tiles.

In [ ]:
def compute_positions(length, tile_size, overlap):
    step = max(tile_size - overlap, 1)
    if length <= tile_size:
        return [0]

    positions = list(range(0, length - tile_size + 1, step))
    if positions[-1] != length - tile_size:
        positions.append(length - tile_size)
    return positions


def make_tiles(image_hwc, tile_size=640, overlap=96):
    height, width, _ = image_hwc.shape
    y_positions = compute_positions(height, tile_size, overlap)
    x_positions = compute_positions(width, tile_size, overlap)

    tiles = []
    for y0 in y_positions:
        for x0 in x_positions:
            y1 = min(y0 + tile_size, height)
            x1 = min(x0 + tile_size, width)
            tile = image_hwc[y0:y1, x0:x1]
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1, "image": tile})
    return tiles

## Display a few example tiles

Before running the model, it is useful to inspect a few tiles and make sure they look reasonable.

In [ ]:
def plot_tiles(image_hwc, tiles, max_tiles=4):
    n = min(len(tiles), max_tiles)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, tile in zip(axes, tiles[:n]):
        ax.imshow(tile["image"])
        ax.set_title(f"x={tile['x0']} y={tile['y0']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


tiles = make_tiles(rgb_display, tile_size=640, overlap=96)
print("Number of tiles:", len(tiles))
plot_tiles(rgb_display, tiles, max_tiles=4)

## Fifth set of helper functions: run YOLO on each tile

This is the main detection function. It loops through the tiles, runs the pretrained model, and stores the results in a pandas table.

A few details matter here:
- YOLO expects image-like values, so we convert each tile from `0-1` floats to `0-255` unsigned integers
- each detection gives us a class, a confidence score, and a bounding box
- we shift each box back into the full-image coordinate system using the tile offsets


In [ ]:
def run_yolo_on_tiles(model, tiles, conf=0.05, imgsz=640):
    rows = []

    for tile_id, tile in enumerate(tiles):
        tile_img = (tile["image"] * 255).clip(0, 255).astype("uint8")
        result = model.predict(tile_img, conf=conf, imgsz=imgsz, verbose=False)[0]
        boxes = result.boxes

        if boxes is None or len(boxes) == 0:
            continue

        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            class_id = int(box.cls[0].cpu().numpy())
            score = float(box.conf[0].cpu().numpy())
            label = model.names[class_id]

            rows.append({
                "tile_id": tile_id,
                "label": label,
                "confidence": score,
                "xmin": x1 + tile["x0"],
                "ymin": y1 + tile["y0"],
                "xmax": x2 + tile["x0"],
                "ymax": y2 + tile["y0"],
            })

    return pd.DataFrame(rows)

## Load the pretrained YOLO model

We use a small pretrained model called `yolov8n.pt`. It is compact and convenient for classroom use.

In [ ]:
!wget https://raw.githubusercontent.com/edwardoughton/satellite-image-analysis/main/yolov8n.pt
model = YOLO("yolov8n.pt")

## Run detection and inspect the raw table

We now run the detector on all tiles. We focus here only on aircraft.

In [ ]:
interesting_labels = {"airplane"}

detections = run_yolo_on_tiles(model, tiles, conf=0.05, imgsz=640)

if detections.empty:
    print("No detections were returned. Try a different site or lower the confidence threshold.")
else:
    detections = detections[detections["label"].isin(interesting_labels)].copy()
    print("Relevant detections:", len(detections))
    display(detections.sort_values("confidence", ascending=False).head(15))

## Plot the detections on the image

This final helper draws each bounding box on top of the original image.

In [ ]:
def plot_detections(image_hwc, detections, keep_labels=None, figsize=(10, 10)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image_hwc)

    if keep_labels is not None:
        detections = detections[detections["label"].isin(keep_labels)].copy()

    colors = {
        "car": "yellow",
        "truck": "orange",
        "bus": "cyan",
        "boat": "deepskyblue",
        "airplane": "lime",
    }

    for _, row in detections.iterrows():
        width = row["xmax"] - row["xmin"]
        height = row["ymax"] - row["ymin"]
        color = colors.get(row["label"], "red")

        rect = patches.Rectangle(
            (row["xmin"], row["ymin"]),
            width,
            height,
            linewidth=1.5,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)

        ax.text(
            row["xmin"],
            max(0, row["ymin"] - 4),
            f"{row['label']} {row['confidence']:.2f}",
            color="white",
            fontsize=8,
            bbox=dict(facecolor=color, alpha=0.7, pad=1),
        )

    ax.set_title("YOLO detections")
    ax.axis("off")
    plt.show()


if not detections.empty:
    print(detections["label"].value_counts())
    plot_detections(rgb_display, detections, keep_labels=interesting_labels, figsize=(12, 12))

## What should students notice?

Questions for discussion:

1. Which classes are detected most confidently?
2. Are there false positives?
3. Why might a model trained on everyday photographs still detect some airplanes or vehicles in aerial imagery?

Key point: a pretrained model can be useful, but only when its training classes and visual perspective are close enough to the new task.

In [ ]:
# Example
# Step 1: Pick area of interest
site_name = "Pinal Airpark, AZ"
bbox = (-111.334, 32.5027, -111.321151, 32.51)

# Step 2: rerun the workflow
item = get_latest_naip_item(bbox)
rgb = read_naip_clip(item, bbox)
rgb_display = stretch_rgb(rgb)
tiles = make_tiles(rgb_display, tile_size=640, overlap=96)
detections = run_yolo_on_tiles(model, tiles, conf=0.05, imgsz=640)

if detections.empty:
    print("No detections returned for this site. Try another bbox or a lower confidence threshold.")
else:
    detections = detections[detections["label"].isin(interesting_labels)].copy()
    print("Site:", site_name)
    print("Image date:", item.datetime.date())
    print("Detections by class:")
    print(detections["label"].value_counts())
    show_image(rgb_display, f"New study area: {site_name}")
    plot_detections(rgb_display, detections, keep_labels=interesting_labels, figsize=(12, 12))

In [ ]:
# Example
# Step 1: Pick area of interest
site_name = "Southern California Logistics Airport, CA"
bbox = (-117.382789, 34.608481, -117.374126, 34.614054)

# Step 2: rerun the workflow
item = get_latest_naip_item(bbox)
rgb = read_naip_clip(item, bbox)
rgb_display = stretch_rgb(rgb)
tiles = make_tiles(rgb_display, tile_size=640, overlap=96)
detections = run_yolo_on_tiles(model, tiles, conf=0.05, imgsz=640)

if detections.empty:
    print("No detections returned for this site. Try another bbox or a lower confidence threshold.")
else:
    detections = detections[detections["label"].isin(interesting_labels)].copy()
    print("Site:", site_name)
    print("Image date:", item.datetime.date())
    print("Detections by class:")
    print(detections["label"].value_counts())
    show_image(rgb_display, f"New study area: {site_name}")
    plot_detections(rgb_display, detections, keep_labels=interesting_labels, figsize=(12, 12))

In [ ]:
# Example
# Step 1: Pick area of interest
site_name = "Mojave Air & Space Port, CA"
bbox = (-118.1518, 35.0617, -118.1411, 35.0678)

# Step 2: rerun the workflow
item = get_latest_naip_item(bbox)
rgb = read_naip_clip(item, bbox)
rgb_display = stretch_rgb(rgb)
tiles = make_tiles(rgb_display, tile_size=640, overlap=96)
detections = run_yolo_on_tiles(model, tiles, conf=0.05, imgsz=640)

if detections.empty:
    print("No detections returned for this site. Try another bbox or a lower confidence threshold.")
else:
    detections = detections[detections["label"].isin(interesting_labels)].copy()
    print("Site:", site_name)
    print("Image date:", item.datetime.date())
    print("Detections by class:")
    print(detections["label"].value_counts())
    show_image(rgb_display, f"New study area: {site_name}")
    plot_detections(rgb_display, detections, keep_labels=interesting_labels, figsize=(12, 12))

## Guided exercise: apply the workflow to a new set of places of interest

Now students should look to reuse the code for new contexts. This is the main class exercise. This pretrained model will have varying degrees of success, given it has not been specifically trained on satellite imagery.

Evaluate the effectiveness on:

- Ships and maritime objects
- Vehicles in parking lots
- Buildings
- Transportation infrastructure
- Energy infrastructure
- Natural objects

The task is to change the geographic bounding box to get images for a new area, and then rerun the workflow.

Write 3 to 5 sentences describing what the model detected in each scene you pick, what it missed, and provide an explanation for why it might have performed certain ways.

## Takeaways

- We imported packages in stages so their roles were clear.
- We used geospatial packages to find, clip, and display a NAIP image.
- We used YOLO to detect a small set of object classes with a pretrained model.
- We used the same code pattern again in various new places to evaluate model effectiveness.